In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
MIN_CLASS_SIZE = 10

In [4]:
df = pd.read_csv("netflix_titles.csv")
df = df.drop_duplicates(subset="title").reset_index(drop=True)

print("Rating category distribution:")
counts = df["rating"].value_counts()
print(counts)


Rating category distribution:
rating
TV-MA       3205
TV-14       2155
TV-PG        861
R            798
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [5]:
rare_ratings = counts[counts < MIN_CLASS_SIZE].index.tolist()
print(f"\nDropping rare rating classes (fewer than {MIN_CLASS_SIZE} titles): {rare_ratings}")
df = df[~df["rating"].isin(rare_ratings)].reset_index(drop=True)


Dropping rare rating classes (fewer than 10 titles): ['TV-Y7-FV', 'NC-17', 'UR']


In [6]:
for col in ["country", "listed_in", "duration", "director"]:
    df[col] = df[col].fillna("Unknown")

In [7]:
df["duration_value"] = df["duration"].str.extract(r"(\d+)").astype(float)
df["duration_value"] = df["duration_value"].fillna(df["duration_value"].median())

In [8]:
df["genre_count"] = df["listed_in"].apply(lambda x: len(x.split(",")))

In [9]:
df["primary_country"] = df["country"].apply(lambda x: x.split(",")[0].strip())
df["primary_genre"] = df["listed_in"].apply(lambda x: x.split(",")[0].strip())

In [10]:
df["has_director"] = (df["director"] != "Unknown").astype(int)

In [11]:
feature_cols = ["type", "primary_country", "primary_genre", "release_year",
                 "duration_value", "genre_count", "has_director"]
X = df[feature_cols].copy()

In [12]:
encoders = {}
for col in ["type", "primary_country", "primary_genre"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

In [13]:
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df["rating"])

In [14]:
print("Feature matrix shape:", X.shape)
print("Rating classes:", list(target_encoder.classes_))
X.head()


Feature matrix shape: (8775, 7)
Rating classes: ['G', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7']


,type,primary_country,primary_genre,release_year,duration_value,genre_count,has_director
0,0,80,10,2020,90.0,1,1
1,1,20,8,2021,1.0,3,1
2,1,80,31,2021,1.0,3,1
3,0,6,4,2021,91.0,2,1
4,0,80,12,1993,125.0,3,1


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [18]:
print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 7020 Test size: 1755


In [19]:
dt = DecisionTreeClassifier(random_state=42, max_depth=10)
dt.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=10, random_state=42)

In [20]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [21]:
for name, model in [("Decision Tree", dt), ("Random Forest", rf)]:
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds,
          labels=range(len(target_encoder.classes_)),
          target_names=target_encoder.classes_, zero_division=0))


--- Decision Tree ---
Accuracy: 0.4877
              precision    recall  f1-score   support

           G       0.38      0.38      0.38         8
          NR       0.00      0.00      0.00        16
          PG       0.63      0.51      0.56        57
       PG-13       0.35      0.31      0.33        98
           R       0.41      0.40      0.40       160
       TV-14       0.47      0.45      0.46       431
        TV-G       0.14      0.02      0.04        44
       TV-MA       0.56      0.71      0.62       641
       TV-PG       0.29      0.14      0.19       172
        TV-Y       0.47      0.52      0.50        61
       TV-Y7       0.42      0.36      0.39        67

    accuracy                           0.49      1755
   macro avg       0.37      0.34      0.35      1755
weighted avg       0.46      0.49      0.47      1755


--- Random Forest ---
Accuracy: 0.4798
              precision    recall  f1-score   support

           G       0.75      0.75      0.75         

In [23]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 3],
}

In [24]:
grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_grid, cv=3, scoring="accuracy", n_jobs=-1,
)
grid.fit(X_train, y_train)
print("Best params found:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)

Best params found: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Best cross-validation accuracy: 0.4874643874643875


In [25]:
tuned_rf = grid.best_estimator_

In [26]:
preds = tuned_rf.predict(X_test)
tuned_acc = accuracy_score(y_test, preds)
print(f"Tuned Random Forest accuracy: {tuned_acc:.4f}")
print(classification_report(y_test, preds,
      labels=range(len(target_encoder.classes_)),
      target_names=target_encoder.classes_, zero_division=0))

print("=== Model Comparison ===")
dt_acc = accuracy_score(y_test, dt.predict(X_test))
rf_acc = accuracy_score(y_test, rf.predict(X_test))
for name, acc in sorted(
    [("Decision Tree", dt_acc), ("Random Forest", rf_acc), ("Tuned Random Forest", tuned_acc)],
    key=lambda x: x[1], reverse=True):
    print(f"{name}: {acc:.4f}")

Tuned Random Forest accuracy: 0.4798
              precision    recall  f1-score   support

           G       0.75      0.75      0.75         8
          NR       0.00      0.00      0.00        16
          PG       0.58      0.58      0.58        57
       PG-13       0.33      0.27      0.29        98
           R       0.43      0.43      0.43       160
       TV-14       0.47      0.46      0.47       431
        TV-G       0.06      0.05      0.05        44
       TV-MA       0.57      0.62      0.60       641
       TV-PG       0.28      0.26      0.27       172
        TV-Y       0.48      0.51      0.49        61
       TV-Y7       0.46      0.46      0.46        67

    accuracy                           0.48      1755
   macro avg       0.40      0.40      0.40      1755
weighted avg       0.47      0.48      0.47      1755

=== Model Comparison ===
Decision Tree: 0.4877
Random Forest: 0.4798
Tuned Random Forest: 0.4798


In [32]:
type_in = input("Type (Movie or TV Show): ")
country_in = input("Country (e.g. United States): ")
genre_in = input("Primary genre (e.g. Dramas): ")
year_in = int(input("Release year: "))
duration_in = float(input("Duration number (min or seasons): "))
genre_count_in = int(input("Number of genre tags: "))
has_director_in = int(input("Has a listed director? (1=yes, 0=no): "))

def safe_encode(encoder, value):
    try:
        return encoder.transform([value])[0]
    except ValueError:
        return 0

sample = pd.DataFrame([{
    "type": safe_encode(encoders["type"], type_in),
    "primary_country": safe_encode(encoders["primary_country"], country_in),
    "primary_genre": safe_encode(encoders["primary_genre"], genre_in),
    "release_year": year_in,
    "duration_value": duration_in,
    "genre_count": genre_count_in,
    "has_director": has_director_in,
}])

pred = tuned_rf.predict(sample)[0]
print(f"\nPredicted rating: {target_encoder.inverse_transform([pred])[0]}")

Type (Movie or TV Show): movie
Country (e.g. United States): United States
Primary genre (e.g. Dramas): Dramas
Release year: 2020
Duration number (min or seasons): 90
Number of genre tags: 4
Has a listed director? (1=yes, 0=no): 1

Predicted rating: TV-MA
